# 源域与目标域数据准备

根据相似性分析结果，选择最佳源域-目标域组合，进行数据分割和标准化。

标准化策略：每个域独立计算自己的 scaler，验证集和测试集复用训练集的 scaler

In [1]:
import pandas as pd

# 从 src 模块导入共享配置和函数
from src import (
    load_data,
    prepare_source_domain,
    prepare_target_domain,
    BASE_DIR,
    BUILDINGS_DIR,
    SCALER_DIR,
    SOURCE_BUILDING,
    TARGET_BUILDING,
    TARGET_SAMPLE_START,
    TARGET_SAMPLE_END,
)

## 加载特征数据

In [2]:
# 加载源域特征数据（全量）
source_path = BUILDINGS_DIR / f'{SOURCE_BUILDING}_特征.csv'
source_df = load_data(source_path)
print(f"源域建筑数据: {SOURCE_BUILDING} {source_df.shape}")

# 加载目标域特征数据（全量）
target_path = BUILDINGS_DIR / f'{TARGET_BUILDING}_特征.csv'
target_df = load_data(target_path)
print(f"目标域建筑数据: {TARGET_BUILDING} {target_df.shape}")
print(f"时间段: {TARGET_SAMPLE_START} ~ {TARGET_SAMPLE_END}")

源域建筑数据: 南阶 (8784, 46)
目标域建筑数据: 环工 (8784, 46)
时间段: 2024-03-15 ~ 2024-06-15


## 处理源域数据

源域数据丰富，不分割数据集，用于预训练模型。全量数据标准化后保存。

In [3]:
source_result = prepare_source_domain(
    df=source_df,
    scaler_dir=SCALER_DIR,
)

source_train_df = source_result['df']
source_load_scaler = source_result['load_scaler']

## 处理目标域数据（选取小样本时间段后分割）

目标域模拟"小样本"场景，选取指定时间段后按比例分割为训练/验证/测试集。

In [4]:
target_result = prepare_target_domain(
    df=target_df,
    sample_start=TARGET_SAMPLE_START,
    sample_end=TARGET_SAMPLE_END,
    scaler_dir=SCALER_DIR,
)

target_train_df = target_result['train_df']
target_val_df = target_result['val_df']
target_test_df = target_result['test_df']
target_load_scaler = target_result['load_scaler']

## 保存标准化数据

In [5]:
# 保存源域数据（全量）
source_train_df.to_csv(BASE_DIR / 'source_train_std.csv', index=False, encoding='utf-8-sig')

# 保存目标域数据
target_train_df.to_csv(BASE_DIR / 'target_train_std.csv', index=False, encoding='utf-8-sig')
target_val_df.to_csv(BASE_DIR / 'target_val_std.csv', index=False, encoding='utf-8-sig')
target_test_df.to_csv(BASE_DIR / 'target_test_std.csv', index=False, encoding='utf-8-sig')

## 数据汇总

In [6]:
summary_data = [
    {
        '角色': '源域',
        '建筑名称': SOURCE_BUILDING,
        '样本数': source_train_df.shape[0],
    },
    {
        '角色': '目标域',
        '建筑名称': TARGET_BUILDING,
        '样本数': target_train_df.shape[0]+target_val_df.shape[0]+target_test_df.shape[0],
        '训练集': target_train_df.shape[0],
        '验证集': target_val_df.shape[0],
        '测试集': target_test_df.shape[0],
    },
]

summary_df = pd.DataFrame(summary_data)
summary_df.set_index(summary_df.columns[0], inplace=True)
summary_df

,建筑名称,样本数,训练集,验证集,测试集
角色,,,,,
源域,南阶,8784,NaN,NaN,NaN
目标域,环工,2208,1545.0,331.0,332.0


## 输出文件说明

| 文件 | 说明 |
|------|------|
| `source_train_std.csv` | 源域全量标准化数据（用于预训练） |
| `target_train_std.csv` | 目标域训练集（小样本的70%） |
| `target_val_std.csv` | 目标域验证集（小样本的15%） |
| `target_test_std.csv` | 目标域测试集（小样本的15%） |
| `scalers/source_load_scaler.joblib` | 源域负荷 scaler |
| `scalers/target_load_scaler.joblib` | 目标域负荷 scaler |